In [ ]:
import os
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Activation, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Concatenate, Dense, Dropout, Reshape
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from gensim.models import Word2Vec

# --- 1. MIXED PRECISION (Gyorsítás) ---
mixed_precision.set_global_policy('mixed_float16')

# --- ÚTVONALAK ---
H5_PATH   = "/content/spotify_dataset_compressed.h5"
W2V_PATH  = "/content/song2vec.model"
SAVE_PATH = "/content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only_markov.keras"

# ==========================================
# 2. EGYÉNI RÉTEGEK ÉS LOSS
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    """L2 normalizálás a kimeneti vektoron a koszinusz-hasonlósághoz."""
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    """Koszinusz távolság alapú veszteségfüggvény."""
    # Mivel y_pred már normalizált az L2NormLayer miatt, csak y_true-t kell normalizálni
    y_true_norm = tf.math.l2_normalize(y_true, axis=1)
    return 1.0 - tf.reduce_mean(tf.reduce_sum(y_true_norm * y_pred, axis=1))

# ==========================================
# 3. DATA GENERATOR (MEL + MARKOV -> W2V)
# ==========================================
class SpotifyHybridGenerator(Sequence):
    def __init__(self, h5_path, w2v_model, split_name='train', batch_size=64, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.h5_path = h5_path
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.w2v_model = w2v_model

        with h5py.File(self.h5_path, 'r') as hf:
            splits = hf['ml/split'][:]
            splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])
            self.indices = np.where(splits_str == split_name)[0]

        print(f"✅ Generátor '{split_name}': {len(self.indices)} dal (Batch size: {batch_size}).")
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = np.sort(self.indices[index * self.batch_size : (index + 1) * self.batch_size])

        with h5py.File(self.h5_path, 'r') as hf:
            # Mel-spektrogram beolvasása
            X_mel = hf['spectrograms/mel'][batch_indices]
            X_mel = np.expand_dims(X_mel, axis=-1).astype(np.float32)

            # Markov-vektor beolvasása (FONTOS: Ellenőrizd a kulcsot a H5-ben!)
            X_markov = hf['features/markov_chords'][batch_indices].astype(np.float32)

            # URI-k a Word2Vec-hez
            uris = hf['ml/track_uri'][batch_indices]

        # Word2Vec célpontok (y)
        y_batch = np.zeros((len(batch_indices), 400), dtype=np.float32)
        for i, uri_bytes in enumerate(uris):
            uri = uri_bytes.decode('utf-8') if isinstance(uri_bytes, bytes) else uri_bytes
            if uri in self.w2v_model.wv:
                y_batch[i] = self.w2v_model.wv[uri]

        # Két bemenet a hálónak
        return {'mel_input': X_mel, 'markov_input': X_markov}, y_batch

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# ==========================================
# 4. MODELL ÉPÍTŐKOCKÁK
# ==========================================
def create_cnn_branch(x, filters, pool_sizes, branch_name):
    """Konvolúciós ág a spektrogramhoz."""
    for i, f in enumerate(filters):
        x = Conv2D(f, kernel_size=(3, 3), padding='same', name=f'{branch_name}_conv_{i+1}')(x)
        x = BatchNormalization(name=f'{branch_name}_bn_{i+1}')(x)
        x = Activation('relu', name=f'{branch_name}_relu_{i+1}')(x)
        x = MaxPooling2D(pool_size=pool_sizes[i], name=f'{branch_name}_pool_{i+1}')(x)

    avg_pool = GlobalAveragePooling2D(name=f'{branch_name}_global_avg')(x)
    max_pool = GlobalMaxPooling2D(name=f'{branch_name}_global_max')(x)
    return Concatenate(name=f'{branch_name}_pool_concat')([avg_pool, max_pool])

# ==========================================
# 5. MODELL ÖSSZEÁLLÍTÁSA
# ==========================================
print("\n🚀 Hibrid Modell építése (Mel + Markov -> Word2Vec)...")

# Bemenetek
input_mel = Input(shape=(128, 1280, 1), name='mel_input')
input_markov = Input(shape=(576,), name='markov_input')

# --- 1. Audio ág (CNN) ---
# Eredmény: 512 elemű jellemzővektor
branch_mel = create_cnn_branch(input_mel, [32, 64, 128, 256], [(2,4), (2,4), (2,2), (2,2)], "mel")

# --- 2. Markov ág (Dense) ---
# Eredmény: 128 elemű zenei szerkezeti vektor
m = Dense(256, activation='relu', name='markov_dense_1')(input_markov)
m = BatchNormalization()(m)
m = Dropout(0.2)(m)
branch_markov = Dense(128, activation='relu', name='markov_dense_2')(m)

# --- 3. Fúzió és Regressziós fej ---
# Összefűzés: 512 + 128 = 640 elem
merged = Concatenate(name='fusion_layer')([branch_mel, branch_markov])

z = Dense(1024, activation='relu', name='dense_1')(merged)
z = BatchNormalization(name='bn_dense_1')(z)
z = Dropout(0.4, name='dropout_1')(z)

z = Dense(512, activation='relu', name='dense_2')(z)
z = BatchNormalization(name='bn_dense_2')(z)
z = Dropout(0.3, name='dropout_2')(z)

# Kimeneti réteg (400 dimenziós Word2Vec vektor)
output_raw = Dense(400, activation='linear', name='word2vec_output')(z)
output_layer = L2NormLayer(name='l2_norm')(output_raw)

# Modell létrehozása
model = Model(inputs=[input_mel, input_markov], outputs=output_layer, name='hybrid_markov_w2v')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=cosine_loss,
    metrics=['mae']
)

# ==========================================
# 6. TANÍTÁS
# ==========================================
print("\n📚 Word2Vec betöltése...")
w2v_model = Word2Vec.load(W2V_PATH)

print("\n⚙️ Generátorok inicializálása...")
train_gen = SpotifyHybridGenerator(H5_PATH, w2v_model, split_name='train', batch_size=64)
val_gen   = SpotifyHybridGenerator(H5_PATH, w2v_model, split_name='val',   batch_size=64, shuffle=False)

# Ha már van korábbi mentés, töltsük be (opcionális)
if os.path.exists(SAVE_PATH):
    print(f"\n💾 Korábbi állapot betöltése: {SAVE_PATH}")
    model = tf.keras.models.load_model(
        SAVE_PATH,
        custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

callbacks = [
    ModelCheckpoint(SAVE_PATH, save_best_only=True, monitor='val_loss', mode='min', verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-5)
]

print("\n🔥 HIBRID TANÍTÁS INDÍTÁSA...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks
)

print(f"\n✅ Tanítás kész! Modell mentve: {SAVE_PATH}")


🚀 Hibrid Modell építése (Mel + Markov -> Word2Vec)...

📚 Word2Vec betöltése...

⚙️ Generátorok inicializálása...
✅ Generátor 'train': 21642 dal (Batch size: 64).
✅ Generátor 'val': 2705 dal (Batch size: 64).

🔥 HIBRID TANÍTÁS INDÍTÁSA...
Epoch 1/15
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.6629 - mae: 0.1599
Epoch 1: val_loss improved from None to 0.57877, saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only_markov.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only_markov.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 850s 2s/step - loss: 0.5469 - mae: 0.1551 - val_loss: 0.5788 - val_mae: 0.1565 - learning_rate: 0.0010
Epoch 2/15
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.4395 - mae: 0.1503
Epoch 2: val_loss improved from 0.57877 to 0.53809, saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only_markov.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only

In [ ]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm

# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
H5_PATH = "../Dataset/spotify_dataset_compressed.h5"
W2V_PATH = "../Models/song2vec.model"
# --- ÚJ: Itt add meg a hibrid (Mel+Markov) modell útvonalát! ---
CNN_PATH = "../Models/spotify_cnn_mel_only_markov.keras"
K_VALUES = [10, 20]
NUM_TEST_SAMPLES = 500

# Custom rétegek és loss a CNN betöltéséhez
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))

    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0

    return hit_rate, precision, recall

def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n2. Ground Truth szótárak építése a RAM-ban (ez eltarthat 1-2 percig)...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)

    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]

        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]

            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz kinyerése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]

        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            np.random.seed(42)
            test_indices = np.random.choice(test_indices, NUM_TEST_SAMPLES, replace=False)

        print(f"Tesztelésre kiválasztva: {len(test_indices)} dal.")

        results = {"baseline": {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES},
                   "cnn":      {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}

        print("\n4. Kiértékelés futtatása...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri

            if uri not in w2v_model.wv:
                continue

            pids = track_to_playlists.get(uri, set())
            if not pids:
                continue

            relevant_uris = set()
            for pid in pids:
                relevant_uris.update(playlist_to_tracks[pid])
            relevant_uris.discard(uri)

            if not relevant_uris:
                continue

            # --- A) BASELINE ---
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL PREDIKCIÓ ---
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)

            # --- ÚJ: Markov vektor beolvasása és formázása ---
            markov = np.expand_dims(hf["features/markov_chords"][idx], axis=0).astype(np.float32)

            # --- ÚJ: Predikció a két bemenettel (Szótár alapú átadás) ---
            pred_vector = cnn_model.predict({'mel_input': mel, 'markov_input': markov}, verbose=0)[0]

            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES))
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]

            if len(cnn_recs) < max(K_VALUES):
                extra = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
                cnn_recs = [u for u, score in extra if u != uri][:max(K_VALUES)]

            # --- METRIKÁK KISZÁMÍTÁSA ---
            for k in K_VALUES:
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)

                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)

    # --- 5. EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*50)
    print("🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆")
    print("="*50)
    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print("BASELINE (Tiszta Word2Vec - Felső korlát):")
        print(f"  Hit Rate@{k}:  {np.mean(results['baseline'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['baseline'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['baseline'][k]['rec'])*100:.2f}%")

        print("\nCNN HIBRID MODELL (Audio + Markov):")
        print(f"  Hit Rate@{k}:  {np.mean(results['cnn'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['cnn'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['cnn'][k]['rec'])*100:.2f}%")

if __name__ == "__main__":
    main()

1. Modellek betöltése...


2. Ground Truth szótárak építése a RAM-ban (ez eltarthat 1-2 percig)...


c:\Users\Béres Gábor\music_recommender\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 66 variables whereas the saved optimizer has 70 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:05<00:00, 530384.56it/s]



3. Teszt halmaz kinyerése...
Tesztelésre kiválasztva: 500 dal.

4. Kiértékelés futtatása...


Dalok tesztelése: 100%|██████████| 500/500 [01:31<00:00,  5.47it/s]



🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆

--- TOP 10 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@10:  98.60%
  Precision@10: 85.00%
  Recall@10:    0.10%

CNN HIBRID MODELL (Audio + Markov):
  Hit Rate@10:  52.20%
  Precision@10: 17.38%
  Recall@10:    0.01%

--- TOP 20 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@20:  99.40%
  Precision@20: 81.43%
  Recall@20:    0.19%

CNN HIBRID MODELL (Audio + Markov):
  Hit Rate@20:  62.40%
  Precision@20: 17.26%
  Recall@20:    0.02%


# Új Markov mátrixokkal és entrópiával

In [ ]:
import os
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Activation, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Concatenate, Dense, Dropout, Reshape, Multiply
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from gensim.models import Word2Vec

# --- ÚTVONALAK ---
H5_PATH   = "/content/spotify_dataset_lite.h5"
W2V_PATH  = "/content/song2vec.model"
SAVE_PATH = "/content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only_markov.keras"

# ==========================================
# 2. EGYÉNI RÉTEGEK ÉS LOSS
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    y_true_norm = tf.math.l2_normalize(y_true, axis=1)
    return 1.0 - tf.reduce_mean(tf.reduce_sum(y_true_norm * y_pred, axis=1))

# ==========================================
# 3. DATA GENERATOR
# ==========================================
class SpotifyHybridGenerator(Sequence):
    def __init__(self, h5_path, w2v_model, split_name='train', batch_size=64, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.h5_path = h5_path
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.w2v_model = w2v_model
        w2v_keys = set(self.w2v_model.wv.key_to_index.keys())

        with h5py.File(self.h5_path, 'r') as hf:
            splits = hf['ml/split'][:]
            splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])
            all_indices = np.where(splits_str == split_name)[0]
            uris = hf['ml/track_uri'][:]

            loc_dim  = hf['features/markov_entropy_local'].shape[1]
            glob_dim = hf['features/markov_entropy_global'].shape[1]
            self.entropy_dim = loc_dim + glob_dim

            valid_indices = [idx for idx in all_indices
                             if (uris[idx].decode('utf-8') if isinstance(uris[idx], bytes) else uris[idx]) in w2v_keys]
            self.indices = np.array(valid_indices)

            # Fix normalizációs statisztikák a TELJES train/val seten (nem batch-szinten)
            all_loc  = hf['features/markov_entropy_local'][self.indices]
            all_glob = hf['features/markov_entropy_global'][self.indices]
            all_ent  = np.concatenate([all_loc, all_glob], axis=1).astype(np.float32)
            self.ent_mean = all_ent.mean(axis=0)
            self.ent_std  = all_ent.std(axis=0)

        print(f"✅ Generátor '{split_name}': {len(self.indices)} dal | Entrópia dim: {self.entropy_dim}")
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = np.sort(self.indices[index * self.batch_size : (index + 1) * self.batch_size])
        with h5py.File(self.h5_path, 'r') as hf:
            X_mel = hf['spectrograms/mel'][batch_indices]
            X_mel = np.expand_dims(X_mel, axis=-1).astype(np.float32)

            X_markov = hf['features/markov_chords'][batch_indices].astype(np.float32)
            X_markov = X_markov / (X_markov.sum(axis=1, keepdims=True) + 1e-6)

            X_ent_loc  = hf['features/markov_entropy_local'][batch_indices].astype(np.float32)
            X_ent_glob = hf['features/markov_entropy_global'][batch_indices].astype(np.float32)
            X_entropy  = np.concatenate([X_ent_loc, X_ent_glob], axis=1)
            # Fix statisztikákkal normalizálás (nem batch-szinten)
            X_entropy  = (X_entropy - self.ent_mean) / (self.ent_std + 1e-6)

            uris = hf['ml/track_uri'][batch_indices]

        y_batch = np.zeros((len(batch_indices), 400), dtype=np.float32)
        for i, uri_bytes in enumerate(uris):
            uri = uri_bytes.decode('utf-8') if isinstance(uri_bytes, bytes) else uri_bytes
            y_batch[i] = self.w2v_model.wv[uri]

        return {'mel_input': X_mel, 'markov_input': X_markov, 'entropy_input': X_entropy}, y_batch

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

# ==========================================
# 4. MODELL ÉPÍTŐKOCKÁK
# ==========================================
def create_cnn_branch(x, filters, pool_sizes, branch_name):
    for i, f in enumerate(filters):
        x = Conv2D(f, kernel_size=(3, 3), padding='same', name=f'{branch_name}_conv_{i+1}')(x)
        x = BatchNormalization(name=f'{branch_name}_bn_{i+1}')(x)
        x = Activation('relu', name=f'{branch_name}_relu_{i+1}')(x)
        x = MaxPooling2D(pool_size=pool_sizes[i], name=f'{branch_name}_pool_{i+1}')(x)
    avg_pool = GlobalAveragePooling2D(name=f'{branch_name}_global_avg')(x)
    max_pool = GlobalMaxPooling2D(name=f'{branch_name}_global_max')(x)
    return Concatenate(name=f'{branch_name}_pool_concat')([avg_pool, max_pool])

# ==========================================
# 5. MODELL FELÉPÍTÉSE
# ==========================================
w2v_model = Word2Vec.load(W2V_PATH)
train_gen = SpotifyHybridGenerator(H5_PATH, w2v_model, split_name='train')
val_gen   = SpotifyHybridGenerator(H5_PATH, w2v_model, split_name='val', shuffle=False)

INITIAL_LR = 1e-4 if os.path.exists(SAVE_PATH) else 1e-3
optimizer  = tf.keras.optimizers.Adam(learning_rate=INITIAL_LR)
loss_fn    = cosine_loss

if os.path.exists(SAVE_PATH):
    print(f"💾 Modell betöltése: {SAVE_PATH}")
    model = tf.keras.models.load_model(
        SAVE_PATH,
        custom_objects={'cosine_loss': loss_fn, 'L2NormLayer': L2NormLayer}
    )
    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['mae'])
else:
    print("🚀 Új Gated Hybrid Modell építése...")
    input_mel     = Input(shape=(128, 1280, 1), name='mel_input')
    input_markov  = Input(shape=(576,), name='markov_input')
    input_entropy = Input(shape=(train_gen.entropy_dim,), name='entropy_input')

    # Mel ág
    m_branch   = create_cnn_branch(input_mel, [32, 64, 128, 256], [(2,4), (2,4), (2,2), (2,2)], "mel")
    branch_mel = Dense(128, activation='relu', name='mel_proj')(m_branch)

    # Markov ág (3 réteg: 24x24 -> 12x12 -> 6x6)
    markov_2d = Reshape((24, 24, 1))(input_markov)
    markov_2d = Conv2D(16, (3,3), padding='same', name='markov_conv_1')(markov_2d)
    markov_2d = BatchNormalization()(markov_2d)
    markov_2d = Activation('relu')(markov_2d)
    markov_2d = MaxPooling2D((2,2))(markov_2d)
    markov_2d = Conv2D(32, (3,3), padding='same', name='markov_conv_2')(markov_2d)
    markov_2d = BatchNormalization()(markov_2d)
    markov_2d = Activation('relu')(markov_2d)
    markov_2d = MaxPooling2D((2,2))(markov_2d)
    markov_2d = Conv2D(64, (3,3), padding='same', name='markov_conv_3')(markov_2d)
    markov_2d = BatchNormalization()(markov_2d)
    markov_2d = Activation('relu')(markov_2d)
    branch_markov_raw = GlobalAveragePooling2D(name='markov_global_avg')(markov_2d)
    branch_markov     = Dense(128, activation='relu', name='markov_proj')(branch_markov_raw)

    # Entrópia ág
    e_branch       = Dense(64)(input_entropy)
    e_branch       = BatchNormalization()(e_branch)
    e_branch       = Activation('relu')(e_branch)
    branch_entropy = Dense(128, activation='relu', name='entropy_proj')(e_branch)

    # Gated Attention
    merged_raw   = Concatenate(name='raw_fusion')([branch_mel, branch_markov, branch_entropy])
    gate_context = Dense(64, activation='relu', name='gate_context')(merged_raw)
    w_mel        = Dense(1, activation='sigmoid', name='gate_mel')(gate_context)
    w_markov     = Dense(1, activation='sigmoid', name='gate_markov')(gate_context)
    w_entropy    = Dense(1, activation='sigmoid', name='gate_entropy')(gate_context)
    gated_mel     = Multiply(name='g_mel')([branch_mel, w_mel])
    gated_markov  = Multiply(name='g_markov')([branch_markov, w_markov])
    gated_entropy = Multiply(name='g_entropy')([branch_entropy, w_entropy])
    merged = Concatenate(name='attention_fusion')([gated_mel, gated_markov, gated_entropy])

    # Regressziós fej
    z = Dense(512, activation='relu')(merged)
    z = BatchNormalization()(z)
    z = Dropout(0.4)(z)
    z = Dense(512, activation='relu')(z)
    z = BatchNormalization()(z)
    z = Dropout(0.3)(z)

    output = Dense(400, activation='linear', name='w2v_output')(z)
    output = L2NormLayer(name='l2_norm')(output)

    model = Model(inputs=[input_mel, input_markov, input_entropy], outputs=output)
    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['mae'])

model.summary()

# ==========================================
# 6. TANÍTÁS
# ==========================================
callbacks = [
    ModelCheckpoint(SAVE_PATH, save_best_only=True, monitor='val_loss', mode='min', verbose=1),
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-5)
]

print("\n🔥 VÉGLEGES HIBRID TANÍTÁS INDÍTÁSA...")
model.fit(train_gen, validation_data=val_gen, epochs=30, callbacks=callbacks)

✅ Generátor 'train': 21642 dal | Entrópia dim: 25
✅ Generátor 'val': 2705 dal | Entrópia dim: 25
🚀 Új Gated Hybrid Modell építése...


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mel_input           │ (None, 128, 1280, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_conv_1 (Conv2D) │ (None, 128, 1280, │        320 │ mel_input[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_bn_1            │ (None, 128, 1280, │        128 │ mel_conv_1[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_relu_1          │ (None, 128, 1280, │          0 │ mel_bn_1[0][0]    │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_pool_1          │ (None, 64, 320,   │          0 │ mel_relu_1[0][0]  │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_conv_2 (Conv2D) │ (None, 64, 320,   │     18,496 │ mel_pool_1[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ markov_input        │ (None, 576)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_bn_2            │ (None, 64, 320,   │        256 │ mel_conv_2[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 24, 24, 1) │          0 │ markov_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_relu_2          │ (None, 64, 320,   │          0 │ mel_bn_2[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ markov_conv_1       │ (None, 24, 24,    │        160 │ reshape_1[0][0]   │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_pool_2          │ (None, 32, 80,    │          0 │ mel_relu_2[0][0]  │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 24, 24,    │         64 │ markov_conv_1[0]… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_conv_3 (Conv2D) │ (None, 32, 80,    │     73,856 │ mel_pool_2[0][0]  │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 24, 24,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_bn_3            │ (None, 32, 80,    │        512 │ mel_conv_3[0][0]  │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 12, 12,    │          0 │ activation_4[0][… │
│ (MaxPooling2D)      │ 16)               │            │                 

 Total params: 1,191,635 (4.55 MB)

 Trainable params: 1,188,275 (4.53 MB)

 Non-trainable params: 3,360 (13.12 KB)


🔥 VÉGLEGES HIBRID TANÍTÁS INDÍTÁSA...
Epoch 1/30
 38/339 ━━━━━━━━━━━━━━━━━━━━ 10:56 2s/step - loss: 0.8732 - mae: 0.1679

KeyboardInterrupt: 

# Arányos dimenzió

In [ ]:
import os
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Activation, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Concatenate, Dense, Dropout, Reshape
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from gensim.models import Word2Vec

# --- ÚTVONALAK ---
H5_PATH   = "/content/spotify_dataset_lite.h5"
W2V_PATH  = "/content/song2vec.model"
SAVE_PATH = "/content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only_markov.keras"

# ==========================================
# 2. EGYÉNI RÉTEGEK ÉS LOSS
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    y_true_norm = tf.math.l2_normalize(y_true, axis=1)
    y_pred_norm = tf.math.l2_normalize(y_pred, axis=1)
    return 1.0 - tf.reduce_mean(tf.reduce_sum(y_true_norm * y_pred_norm, axis=1))

# ==========================================
# 3. DATA GENERATOR
# ==========================================
class SpotifyHybridGenerator(Sequence):
    def __init__(self, h5_path, w2v_model, split_name='train', batch_size=64,
                 shuffle=True, ent_mean=None, ent_std=None, **kwargs):
        super().__init__(**kwargs)
        self.h5_path = h5_path
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.w2v_model = w2v_model
        w2v_keys = set(self.w2v_model.wv.key_to_index.keys())

        with h5py.File(self.h5_path, 'r') as hf:
            splits = hf['ml/split'][:]
            splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])
            all_indices = np.where(splits_str == split_name)[0]
            uris = hf['ml/track_uri'][:]

            loc_dim  = hf['features/markov_entropy_local'].shape[1]
            glob_dim = hf['features/markov_entropy_global'].shape[1]
            self.entropy_dim = loc_dim + glob_dim

            valid_indices = [idx for idx in all_indices
                             if (uris[idx].decode('utf-8') if isinstance(uris[idx], bytes) else uris[idx]) in w2v_keys]
            self.indices = np.array(valid_indices)

            if ent_mean is not None and ent_std is not None:
                self.ent_mean = ent_mean
                self.ent_std  = ent_std
            else:
                all_loc  = hf['features/markov_entropy_local'][self.indices]
                all_glob = hf['features/markov_entropy_global'][self.indices]
                all_ent  = np.concatenate([all_loc, all_glob], axis=1).astype(np.float32)
                self.ent_mean = all_ent.mean(axis=0)
                self.ent_std  = all_ent.std(axis=0)

        print(f"✅ Generátor '{split_name}': {len(self.indices)} dal | Entrópia dim: {self.entropy_dim}")
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = np.sort(self.indices[index * self.batch_size : (index + 1) * self.batch_size])
        with h5py.File(self.h5_path, 'r') as hf:
            X_mel = hf['spectrograms/mel'][batch_indices]
            X_mel = np.expand_dims(X_mel, axis=-1).astype(np.float32)
            mean_mel = np.mean(X_mel, axis=(1, 2, 3), keepdims=True)
            std_mel = np.std(X_mel, axis=(1, 2, 3), keepdims=True)
            X_mel = (X_mel - mean_mel) / (std_mel + 1e-6)

            X_markov = hf['features/markov_chords'][batch_indices].astype(np.float32)
            X_markov = X_markov.reshape(-1, 24, 24)
            row_sums = X_markov.sum(axis=2, keepdims=True)
            X_markov = X_markov / (row_sums + 1e-6)
            X_markov = X_markov.reshape(-1, 576)

            X_ent_loc  = hf['features/markov_entropy_local'][batch_indices].astype(np.float32)
            X_ent_glob = hf['features/markov_entropy_global'][batch_indices].astype(np.float32)
            X_entropy  = np.concatenate([X_ent_loc, X_ent_glob], axis=1)
            X_entropy  = (X_entropy - self.ent_mean) / (self.ent_std + 1e-6)

            uris = hf['ml/track_uri'][batch_indices]

        y_batch = np.zeros((len(batch_indices), 400), dtype=np.float32)
        for i, uri_bytes in enumerate(uris):
            uri = uri_bytes.decode('utf-8') if isinstance(uri_bytes, bytes) else uri_bytes
            y_batch[i] = self.w2v_model.wv[uri]

        y_norms = np.linalg.norm(y_batch, axis=1, keepdims=True)
        y_batch = y_batch / (y_norms + 1e-8)

        return {'mel_input': X_mel, 'markov_input': X_markov, 'entropy_input': X_entropy}, y_batch

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

# ==========================================
# 4. MODELL ÉPÍTŐKOCKÁK
# ==========================================
def create_cnn_branch(x, filters, pool_sizes, branch_name):
    for i, f in enumerate(filters):
        x = Conv2D(f, kernel_size=(3, 3), padding='same', name=f'{branch_name}_conv_{i+1}')(x)
        x = BatchNormalization(name=f'{branch_name}_bn_{i+1}')(x)
        x = Activation('relu', name=f'{branch_name}_relu_{i+1}')(x)
        x = MaxPooling2D(pool_size=pool_sizes[i], name=f'{branch_name}_pool_{i+1}')(x)
    avg_pool = GlobalAveragePooling2D(name=f'{branch_name}_global_avg')(x)
    max_pool = GlobalMaxPooling2D(name=f'{branch_name}_global_max')(x)
    return Concatenate(name=f'{branch_name}_pool_concat')([avg_pool, max_pool])

# ==========================================
# 5. MODELL FELÉPÍTÉSE
# ==========================================
w2v_model = Word2Vec.load(W2V_PATH)

train_gen = SpotifyHybridGenerator(H5_PATH, w2v_model, split_name='train')
val_gen   = SpotifyHybridGenerator(H5_PATH, w2v_model, split_name='val',
                                   ent_mean=train_gen.ent_mean,
                                   ent_std=train_gen.ent_std,
                                   shuffle=False)

INITIAL_LR = 1e-4 if not os.path.exists(SAVE_PATH) else 5e-5
optimizer  = tf.keras.optimizers.Adam(learning_rate=INITIAL_LR)
loss_fn    = cosine_loss

if os.path.exists(SAVE_PATH):
    print(f"💾 Modell betöltése: {SAVE_PATH}")
    model = tf.keras.models.load_model(
        SAVE_PATH,
        custom_objects={'cosine_loss': loss_fn, 'L2NormLayer': L2NormLayer}
    )
    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['mae'])
else:
    print("🚀 Arányos Dimenziójú Hybrid Modell építése (Kapuzás nélkül)...")
    input_mel     = Input(shape=(128, 1280, 1), name='mel_input')
    input_markov  = Input(shape=(576,), name='markov_input')
    input_entropy = Input(shape=(train_gen.entropy_dim,), name='entropy_input')

    # --- Mel ág ---
    m_branch   = create_cnn_branch(input_mel, [32, 64, 128, 256], [(2,4), (2,4), (2,2), (2,2)], "mel")
    branch_mel = Dense(256, activation='relu', name='mel_proj')(m_branch)

    # --- Markov ág ---
    markov_2d = Reshape((24, 24, 1))(input_markov)
    markov_2d = Conv2D(16, (3,3), padding='same', name='markov_conv_1')(markov_2d)
    markov_2d = BatchNormalization()(markov_2d)
    markov_2d = Activation('relu')(markov_2d)
    markov_2d = MaxPooling2D((2,2))(markov_2d)
    markov_2d = Conv2D(32, (3,3), padding='same', name='markov_conv_2')(markov_2d)
    markov_2d = BatchNormalization()(markov_2d)
    markov_2d = Activation('relu')(markov_2d)
    markov_2d = MaxPooling2D((2,2))(markov_2d)
    markov_2d = Conv2D(64, (3,3), padding='same', name='markov_conv_3')(markov_2d)
    markov_2d = BatchNormalization()(markov_2d)
    markov_2d = Activation('relu')(markov_2d)
    branch_markov_raw = GlobalAveragePooling2D(name='markov_global_avg')(markov_2d)
    branch_markov     = Dense(64, activation='relu', name='markov_proj')(branch_markov_raw)

    # --- Entrópia ág ---
    e_branch       = Dense(64)(input_entropy)
    e_branch       = BatchNormalization()(e_branch)
    e_branch       = Activation('relu')(e_branch)
    branch_entropy = Dense(32, activation='relu', name='entropy_proj')(e_branch)

    # --- Egyszerű Fúzió (Gated Attention eltávolítva) ---
    merged = Concatenate(name='simple_fusion')([branch_mel, branch_markov, branch_entropy])

    # --- Stabilizált Regressziós fej ---
    z = Dense(256, name='reg_dense_1')(merged)
    z = BatchNormalization(name='reg_bn_1')(z)
    z = Activation('relu', name='reg_relu_1')(z)
    z = Dropout(0.4, name='reg_drop_1')(z)

    output = Dense(400, activation='linear', name='w2v_output')(z)
    output = L2NormLayer(name='l2_norm')(output)

    model = Model(inputs=[input_mel, input_markov, input_entropy], outputs=output)
    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['mae'])

model.summary()

# ==========================================
# 6. TANÍTÁS
# ==========================================
callbacks = [
    ModelCheckpoint(SAVE_PATH, save_best_only=True, monitor='val_loss', mode='min', verbose=1),
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-6)
]

print("\n🔥 KÖZVETLEN FÚZIÓS HIBRID TANÍTÁS INDÍTÁSA...")
model.fit(train_gen, validation_data=val_gen, epochs=30, callbacks=callbacks)

✅ Generátor 'train': 21642 dal | Entrópia dim: 25
✅ Generátor 'val': 2705 dal | Entrópia dim: 25
🚀 Arányos Dimenziójú Hybrid Modell építése (Kapuzás nélkül)...


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mel_input           │ (None, 128, 1280, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_conv_1 (Conv2D) │ (None, 128, 1280, │        320 │ mel_input[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_bn_1            │ (None, 128, 1280, │        128 │ mel_conv_1[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_relu_1          │ (None, 128, 1280, │          0 │ mel_bn_1[0][0]    │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_pool_1          │ (None, 64, 320,   │          0 │ mel_relu_1[0][0]  │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_conv_2 (Conv2D) │ (None, 64, 320,   │     18,496 │ mel_pool_1[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ markov_input        │ (None, 576)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_bn_2            │ (None, 64, 320,   │        256 │ mel_conv_2[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 24, 24, 1) │          0 │ markov_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_relu_2          │ (None, 64, 320,   │          0 │ mel_bn_2[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ markov_conv_1       │ (None, 24, 24,    │        160 │ reshape_1[0][0]   │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_pool_2          │ (None, 32, 80,    │          0 │ mel_relu_2[0][0]  │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 24, 24,    │         64 │ markov_conv_1[0]… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_conv_3 (Conv2D) │ (None, 32, 80,    │     73,856 │ mel_pool_2[0][0]  │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 24, 24,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_bn_3            │ (None, 32, 80,    │        512 │ mel_conv_3[0][0]  │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 12, 12,    │          0 │ activation_4[0][… │
│ (MaxPooling2D)      │ 16)               │            │                 

 Total params: 747,184 (2.85 MB)

 Trainable params: 745,360 (2.84 MB)

 Non-trainable params: 1,824 (7.12 KB)


🔥 KÖZVETLEN FÚZIÓS HIBRID TANÍTÁS INDÍTÁSA...
Epoch 1/30
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.7478 - mae: 0.0485
Epoch 1: val_loss improved from None to 0.55033, saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only_markov.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only_markov.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 830s 2s/step - loss: 0.6338 - mae: 0.0447 - val_loss: 0.5503 - val_mae: 0.0419 - learning_rate: 1.0000e-04
Epoch 2/30
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.5200 - mae: 0.0406
Epoch 2: val_loss improved from 0.55033 to 0.48402, saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only_markov.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Diplomamunka/spotify_cnn_mel_only_markov.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 802s 2s/step - loss: 0.5111 - mae: 0.0403 - val_loss: 0.4840 - val_mae: 0.0392 - learning_rate: 1.0000e-04
Epoch 3/30
339/339 ━━━━━━━━━━━━━━━━━━━

# Kiértékelés

In [1]:
import os
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm

# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
H5_PATH = "../Dataset/spotify_dataset_lite.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_mel_only_markov.keras" 
K_VALUES = [10, 20, 50, 100, 500] 
NUM_TEST_SAMPLES = 500
STEP_SIZE = 5

# --- EGYÉNI RÉTEGEK ÉS LOSS ---
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    y_true_norm = tf.math.l2_normalize(y_true, axis=1)
    y_pred_norm = tf.math.l2_normalize(y_pred, axis=1)
    return 1.0 - tf.reduce_mean(tf.reduce_sum(y_true_norm * y_pred_norm, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n2. Ground Truth szótárak építése a RAM-ban...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz kinyerése és Train statisztikák számítása...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])
        
        # Train statisztikák az entrópiához (KÖTELEZŐ!)
        train_indices = np.where(splits_str == "train")[0]
        # Csak azokat vesszük, amik benne vannak a W2V-ben
        valid_train_indices = [idx for idx in train_indices 
                               if (hf["ml/track_uri"][idx].decode('utf-8') if isinstance(hf["ml/track_uri"][idx], bytes) else hf["ml/track_uri"][idx]) in w2v_model.wv]
        
        all_loc = hf['features/markov_entropy_local'][valid_train_indices]
        all_glob = hf['features/markov_entropy_global'][valid_train_indices]
        all_ent = np.concatenate([all_loc, all_glob], axis=1).astype(np.float32)
        ent_mean = all_ent.mean(axis=0)
        ent_std = all_ent.std(axis=0)

        # Teszt indexek kinyerése
        all_test_indices = np.where(splits_str == "test")[0]
        test_indices = all_test_indices[::STEP_SIZE]
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            test_indices = test_indices[:NUM_TEST_SAMPLES]
            
        print(f"Tesztelésre kiválasztva: {len(test_indices)} dal.")

        results = {"baseline": {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES},
                   "cnn":      {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}

        print("\n4. Kiértékelés futtatása...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            if uri not in w2v_model.wv: continue
                
            pids = track_to_playlists.get(uri, set())
            if not pids: continue 
                
            relevant_uris = set()
            for pid in pids: relevant_uris.update(playlist_to_tracks[pid])
            relevant_uris.discard(uri) 
            relevant_uris = {u for u in relevant_uris if u in w2v_model.wv}
            
            if not relevant_uris: continue

            # --- A) BASELINE ---
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL PREDIKCIÓ (NORMALIZÁLVA!) ---
            # 1. Mel ág
            mel = hf["spectrograms/mel"][idx]
            mel = np.expand_dims(mel, axis=-1).astype(np.float32)
            mel = np.expand_dims(mel, axis=0) # Batch dimenzió (1, 128, 1280, 1)
            mean_mel = np.mean(mel, axis=(1, 2, 3), keepdims=True)
            std_mel = np.std(mel, axis=(1, 2, 3), keepdims=True)
            mel = (mel - mean_mel) / (std_mel + 1e-6)

            # 2. Markov ág
            markov = hf['features/markov_chords'][idx].astype(np.float32)
            markov = markov.reshape(24, 24)
            row_sums = markov.sum(axis=1, keepdims=True)
            markov = markov / (row_sums + 1e-6)
            markov = markov.reshape(1, 576) # Batch dimenzió (1, 576)

            # 3. Entrópia ág
            ent_loc = hf['features/markov_entropy_local'][idx].astype(np.float32)
            ent_glob = hf['features/markov_entropy_global'][idx].astype(np.float32)
            entropy = np.concatenate([ent_loc, ent_glob])
            entropy = np.expand_dims(entropy, axis=0) # Batch dimenzió (1, D)
            entropy = (entropy - ent_mean) / (ent_std + 1e-6)
            
            pred_vector = cnn_model.predict([mel, markov, entropy], verbose=0)[0]
            
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]

            # --- METRIKÁK ---
            for k in K_VALUES:
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)

    # --- 5. EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*50)
    print("🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆")
    print("="*50)
    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print("BASELINE (Tiszta Word2Vec - Felső korlát):")
        print(f"  Hit Rate@{k}:  {np.mean(results['baseline'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['baseline'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        
        print("\nHIBRID MODELL (Mel + Markov + Entrópia):")
        print(f"  Hit Rate@{k}:  {np.mean(results['cnn'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['cnn'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['cnn'][k]['rec'])*100:.2f}%")

if __name__ == "__main__":
    main()

1. Modellek betöltése...


2. Ground Truth szótárak építése a RAM-ban...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:36<00:00, 422728.06it/s]



3. Teszt halmaz kinyerése és Train statisztikák számítása...
Tesztelésre kiválasztva: 500 dal.

4. Kiértékelés futtatása...


Dalok tesztelése: 100%|██████████| 500/500 [01:43<00:00,  4.82it/s]



🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆

--- TOP 10 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@10:  98.40%
  Precision@10: 84.20%
  Recall@10:    0.15%

HIBRID MODELL (Mel + Markov + Entrópia):
  Hit Rate@10:  48.20%
  Precision@10: 13.88%
  Recall@10:    0.01%

--- TOP 20 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@20:  98.80%
  Precision@20: 80.85%
  Recall@20:    0.27%

HIBRID MODELL (Mel + Markov + Entrópia):
  Hit Rate@20:  58.00%
  Precision@20: 14.07%
  Recall@20:    0.02%

--- TOP 50 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@50:  99.80%
  Precision@50: 75.26%
  Recall@50:    0.61%

HIBRID MODELL (Mel + Markov + Entrópia):
  Hit Rate@50:  68.20%
  Precision@50: 14.26%
  Recall@50:    0.06%

--- TOP 100 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@100:  100.00%
  Precision@100: 70.36%
  Recall@100:    1.08%

HIBRID MODELL (Mel + Markov + Entrópia):
  Hit Rate@100:  76.60%
  Precision@100: 14.26%
  Recall

In [2]:
import os
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm

# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
H5_PATH = "../Dataset/spotify_dataset_lite.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_mel_only_markov_continue.keras" 
K_VALUES = [10, 20, 50, 100, 500] 
NUM_TEST_SAMPLES = 500
STEP_SIZE = 5

# --- EGYÉNI RÉTEGEK ÉS LOSS ---
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    y_true_norm = tf.math.l2_normalize(y_true, axis=1)
    y_pred_norm = tf.math.l2_normalize(y_pred, axis=1)
    return 1.0 - tf.reduce_mean(tf.reduce_sum(y_true_norm * y_pred_norm, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n2. Ground Truth szótárak építése a RAM-ban...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz kinyerése és Train statisztikák számítása...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])
        
        # Train statisztikák az entrópiához (KÖTELEZŐ!)
        train_indices = np.where(splits_str == "train")[0]
        # Csak azokat vesszük, amik benne vannak a W2V-ben
        valid_train_indices = [idx for idx in train_indices 
                               if (hf["ml/track_uri"][idx].decode('utf-8') if isinstance(hf["ml/track_uri"][idx], bytes) else hf["ml/track_uri"][idx]) in w2v_model.wv]
        
        all_loc = hf['features/markov_entropy_local'][valid_train_indices]
        all_glob = hf['features/markov_entropy_global'][valid_train_indices]
        all_ent = np.concatenate([all_loc, all_glob], axis=1).astype(np.float32)
        ent_mean = all_ent.mean(axis=0)
        ent_std = all_ent.std(axis=0)

        # Teszt indexek kinyerése
        all_test_indices = np.where(splits_str == "test")[0]
        test_indices = all_test_indices[::STEP_SIZE]
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            test_indices = test_indices[:NUM_TEST_SAMPLES]
            
        print(f"Tesztelésre kiválasztva: {len(test_indices)} dal.")

        results = {"baseline": {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES},
                   "cnn":      {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}

        print("\n4. Kiértékelés futtatása...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            if uri not in w2v_model.wv: continue
                
            pids = track_to_playlists.get(uri, set())
            if not pids: continue 
                
            relevant_uris = set()
            for pid in pids: relevant_uris.update(playlist_to_tracks[pid])
            relevant_uris.discard(uri) 
            relevant_uris = {u for u in relevant_uris if u in w2v_model.wv}
            
            if not relevant_uris: continue

            # --- A) BASELINE ---
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL PREDIKCIÓ (NORMALIZÁLVA!) ---
            # 1. Mel ág
            mel = hf["spectrograms/mel"][idx]
            mel = np.expand_dims(mel, axis=-1).astype(np.float32)
            mel = np.expand_dims(mel, axis=0) # Batch dimenzió (1, 128, 1280, 1)
            mean_mel = np.mean(mel, axis=(1, 2, 3), keepdims=True)
            std_mel = np.std(mel, axis=(1, 2, 3), keepdims=True)
            mel = (mel - mean_mel) / (std_mel + 1e-6)

            # 2. Markov ág
            markov = hf['features/markov_chords'][idx].astype(np.float32)
            markov = markov.reshape(24, 24)
            row_sums = markov.sum(axis=1, keepdims=True)
            markov = markov / (row_sums + 1e-6)
            markov = markov.reshape(1, 576) # Batch dimenzió (1, 576)

            # 3. Entrópia ág
            ent_loc = hf['features/markov_entropy_local'][idx].astype(np.float32)
            ent_glob = hf['features/markov_entropy_global'][idx].astype(np.float32)
            entropy = np.concatenate([ent_loc, ent_glob])
            entropy = np.expand_dims(entropy, axis=0) # Batch dimenzió (1, D)
            entropy = (entropy - ent_mean) / (ent_std + 1e-6)
            
            pred_vector = cnn_model.predict([mel, markov, entropy], verbose=0)[0]
            
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]

            # --- METRIKÁK ---
            for k in K_VALUES:
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)

    # --- 5. EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*50)
    print("🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆")
    print("="*50)
    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print("BASELINE (Tiszta Word2Vec - Felső korlát):")
        print(f"  Hit Rate@{k}:  {np.mean(results['baseline'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['baseline'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        
        print("\nHIBRID MODELL (Mel + Markov + Entrópia):")
        print(f"  Hit Rate@{k}:  {np.mean(results['cnn'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['cnn'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['cnn'][k]['rec'])*100:.2f}%")

if __name__ == "__main__":
    main()

1. Modellek betöltése...

2. Ground Truth szótárak építése a RAM-ban...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:23<00:00, 463433.45it/s]



3. Teszt halmaz kinyerése és Train statisztikák számítása...
Tesztelésre kiválasztva: 500 dal.

4. Kiértékelés futtatása...


Dalok tesztelése: 100%|██████████| 500/500 [01:53<00:00,  4.40it/s]



🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆

--- TOP 10 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@10:  98.40%
  Precision@10: 84.20%
  Recall@10:    0.15%

HIBRID MODELL (Mel + Markov + Entrópia):
  Hit Rate@10:  50.80%
  Precision@10: 15.18%
  Recall@10:    0.01%

--- TOP 20 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@20:  98.80%
  Precision@20: 80.85%
  Recall@20:    0.27%

HIBRID MODELL (Mel + Markov + Entrópia):
  Hit Rate@20:  58.80%
  Precision@20: 14.98%
  Recall@20:    0.02%

--- TOP 50 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@50:  99.80%
  Precision@50: 75.26%
  Recall@50:    0.61%

HIBRID MODELL (Mel + Markov + Entrópia):
  Hit Rate@50:  70.60%
  Precision@50: 15.05%
  Recall@50:    0.06%

--- TOP 100 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@100:  100.00%
  Precision@100: 70.36%
  Recall@100:    1.08%

HIBRID MODELL (Mel + Markov + Entrópia):
  Hit Rate@100:  77.00%
  Precision@100: 14.95%
  Recall

In [ ]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm 
import random
from sklearn.metrics import roc_auc_score
import copy  # Hozzáadva a hibrid modell klónozásához!

# ==========================================
# 1. ÚTVONALAK ÉS BEÁLLÍTÁSOK
# ==========================================
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_mel_only.keras" 
HYBRID_MATRIX_PATH = "../Models/hybrid_embedding_matrix_mel.npy" # A létrehozott hibrid mátrixod

K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 

# Tesztelési korlátok a gyorsasághoz
NUM_TEST_SAMPLES = 1000  # None, ha az összeset akarod
STEP_SIZE = 2          # Minden hanyadik teszt elemet vegyük

# ==========================================
# 2. SEGÉDFÜGGVÉNYEK ÉS EGYEDI RÉTEGEK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    y_true_norm = tf.math.l2_normalize(y_true, axis=1)
    y_pred_norm = tf.math.l2_normalize(y_pred, axis=1)
    return 1.0 - tf.reduce_mean(tf.reduce_sum(y_true_norm * y_pred_norm, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

# --- AUC SZÁMÍTÁSA (Sampled ROC-AUC) ---
def calculate_auc(query_vector, relevant_uris, w2v_model, num_negatives=1000):
    """
    Megméri, hogy a modell mekkora eséllyel rangsorolja előrébb a 
    valóban releváns dalokat a véletlen negatívokhoz képest.
    """
    # 1. Pozitív minták pontszámai (amikkel már volt közös listán)
    pos_scores = []
    for uri in relevant_uris:
        if uri in w2v_model.wv:
            # Koszinusz hasonlóság (vektorok normáltak, így a dot product elég)
            score = np.dot(query_vector, w2v_model.wv[uri])
            pos_scores.append(score)
    
    if not pos_scores: 
        return 0.5 # Ha nincs pozitív adat, a véletlen szintet adjuk vissza
    
    # 2. Negatív minták (véletlen dalok, amikkel SOHA nem volt közös listán)
    neg_scores = []
    all_uris = list(w2v_model.wv.key_to_index.keys())
    
    # Biztonsági korlát a mintavételhez
    n_neg = min(num_negatives, len(all_uris) - len(relevant_uris))
    
    while len(neg_scores) < n_neg:
        random_uri = random.choice(all_uris)
        if random_uri not in relevant_uris:
            score = np.dot(query_vector, w2v_model.wv[random_uri])
            neg_scores.append(score)
            
    # 3. ROC-AUC kiszámítása a scikit-learn segítségével
    y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
    y_scores = pos_scores + neg_scores
    
    try:
        return roc_auc_score(y_true, y_scores)
    except:
        return 0.5

# ==========================================
# 3. FŐPROGRAM
# ==========================================
def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n[+] HIBRID vektortér betöltése a memóriába...")
    # Klónozzuk a w2v modellt, hogy a szótár (vocab) megmaradjon
    hybrid_w2v = copy.deepcopy(w2v_model)
    
    # Betöltjük a numpy mátrixot
    hybrid_matrix = np.load(HYBRID_MATRIX_PATH)
    
    # FELÜLÍRJUK a klónozott modell vektorait a hibrid mátrixszal!
    hybrid_w2v.wv.vectors = hybrid_matrix
    
    # KRITIKUS LÉPÉS: Újraszámoljuk a normákat a cosine similarity-hez
    hybrid_w2v.wv.fill_norms(force=True) 
    print("Hibrid vektortér sikeresen integrálva!")

    print("\n2. Kapcsolati háló (Ground Truth) építése a memóriában...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz előkészítése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        all_test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]
        
        # Determinisztikus mintavétel
        test_indices = all_test_indices[::STEP_SIZE]
        
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            test_indices = test_indices[:NUM_TEST_SAMPLES]
            
        print(f"Tesztelésre kijelölve: {len(test_indices)} dal.")

        # Eredménytároló szótár (kibővítve a hibriddel)
        results = {
            "baseline": {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
            "cnn":      {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
            "hybrid":   {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}
        }

        print("\n4. Kiértékelés indítása (AUC + Top-K)...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            if uri not in w2v_model.wv:
                continue
                
            # --- RELEVÁNS DALOK (Akikkel már volt egy listán) ---
            pids_of_track = track_to_playlists.get(uri, set())
            if not pids_of_track: continue 
                
            relevant_uris = set()
            for pid in pids_of_track:
                relevant_uris.update(playlist_to_tracks[pid])
            
            relevant_uris.discard(uri) # Önmagát ne ajánlja
            relevant_uris = {u for u in relevant_uris if u in w2v_model.wv} # Csak ami benne van a szótárban
            
            if not relevant_uris: continue

            # --- A) BASELINE (Word2Vec) ---
            baseline_vector = w2v_model.wv[uri]
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL (Audio) ---
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)
            
            pred_vector = cnn_model.predict(mel, verbose=0)[0]
            
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]

            # --- C) HIBRID MODELL ---
            hybrid_similars = hybrid_w2v.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            hybrid_recs = [u for u, score in hybrid_similars if u != uri][:max(K_VALUES)]

            # --- AUC SZÁMÍTÁS ---
            results["baseline"]["auc"].append(calculate_auc(baseline_vector, relevant_uris, w2v_model))
            results["cnn"]["auc"].append(calculate_auc(pred_vector, relevant_uris, w2v_model))
            results["hybrid"]["auc"].append(calculate_auc(pred_vector, relevant_uris, hybrid_w2v))

            # --- TOP-K METRIKÁK SZÁMÍTÁSA ---
            for k in K_VALUES:
                # Baseline
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                # CNN
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)
                
                # Hybrid
                hr_h, prec_h, rec_h = calculate_metrics(hybrid_recs, relevant_uris, k)
                results["hybrid"][k]["hr"].append(hr_h)
                results["hybrid"][k]["prec"].append(prec_h)
                results["hybrid"][k]["rec"].append(rec_h)

    # ==========================================
    # 4. EREDMÉNYEK ÖSSZESÍTÉSE
    # ==========================================
    print("\n" + "="*70)
    print("VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity) 🏆")
    print("="*70)
    
    print(f"\nGLOBÁLIS RANGSOROLÁSI MINŐSÉG:")
    print(f"  BASELINE (Word2Vec) AUC: {np.mean(results['baseline']['auc'])*100:.2f}%")
    print(f"  CNN MODEL (Audio)   AUC: {np.mean(results['cnn']['auc'])*100:.2f}%")
    print(f"  HIBRID MODELL       AUC: {np.mean(results['hybrid']['auc'])*100:.2f}%")

    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print(f"  BASELINE -> HR: {np.mean(results['baseline'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['baseline'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        print(f"  CNN      -> HR: {np.mean(results['cnn'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['cnn'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['cnn'][k]['rec'])*100:.2f}%")
        print(f"  HIBRID   -> HR: {np.mean(results['hybrid'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['hybrid'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['hybrid'][k]['rec'])*100:.2f}%")
    print("\n" + "="*70)

if __name__ == "__main__":
    main()